# 01 · Entrenamiento y evaluación (versión corregida)

Reemplaza a `Transfer_Learning_.ipynb`. La lógica está en `pipeline.py`; cada corrección lleva una etiqueta **[FIX-xx]** explicada en `diferencias_codigo.docx`.

**Principales correcciones**

| | Original | Corregido |
|---|---|---|
| Partición | aleatoria sobre 14.000 imágenes ya aumentadas | por imagen **original**, estratificada; test fijo sin aumentar |
| Augmentation | antes de partir (carpeta `data_aug7`) | al vuelo, **solo** en entrenamiento |
| Backbone | sin congelar en el código; un único objeto compartido | nuevo en cada experimento/trial/semilla, congelado (pesos **y** BatchNorm) |
| Optuna | el batch size sugerido no se usaba | los loaders se pasan a `fit()` |
| Evaluación | última época; F1 ponderado | mejor época en validación; test una vez; 5 semillas; métricas por clase, IC95% |

**Requisito de datos:** la carpeta con las 2.000 imágenes originales (`images_or`) y un CSV con una fila por imagen original (`nombre_imagen;label`, con la emoción en español tal como en `BDD7.csv`).

In [4]:
import os, sys

REPO = 'publicity_analysis'
if not os.path.exists(f'/content/{REPO}'):
    !git clone https://github.com/mlacarrasco/{REPO}.git /content/{REPO}

os.chdir(f'/content/{REPO}')
sys.path.insert(0, f'/content/{REPO}')
!pwd && ls

Cloning into '/content/publicity_analysis'...
remote: Enumerating objects: 2056, done.
remote: Total 2056 (delta 0), reused 0 (delta 0), pack-reused 2056 (from 1)
Receiving objects: 100% (2056/2056), 355.82 MiB | 30.17 MiB/s, done.
Updating files: 100% (2055/2055), done.
/content/publicity_analysis
01_entrenamiento.ipynb	etiquetas_originales.csv  pipeline.py
02_lime.ipynb		images_or		  resultados


In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

import pipeline as pl

pd.set_option('display.float_format', '{:.3f}'.format)

ModuleNotFoundError: No module named 'pipeline'

In [12]:
!pip install numpy pandas torch torchvision pillow scipy scikit-learn optuna

11566.04s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached optuna-5.0.0-py3-none-any.whl.metadata (14 kB)
  Using cached alembic-1.20.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached colorlog-6.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached mako-1.4.3-py3-none-any.whl.metadata (2.9 kB)
Using cached optuna-5.0.0-py3-none-any.whl (442 kB)
Using cached alembic-1.20.0-py3-none-any.whl (268 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.8 MB/s eta 0:00:00
Using cached colorlog-6.12.0-py3-none-any.whl (12 kB)
Using cached mako-1.4.3-py3-none-any.whl (80 kB)
Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl (173 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [optuna]2m6/7 [optuna]]my]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python3.13 -m pip install --upgrade pip

In [3]:
!pwd

/content


## 1. Configuración
Edite aquí las rutas. Todos los parámetros y su justificación están en `pl.Config`.

In [3]:
cfg = pl.Config(
    originals_dir='images_or',
    labels_csv='etiquetas_originales.csv',
    output_dir='resultados',
)
# Experimentos de la ablación (Tabla II) + variantes espaciales [FIX-05]
EXPERIMENTS = [1, 2, 3, 4, 5, 6, 7, 8, '4s', '8s']

os.makedirs(cfg.output_dir, exist_ok=True)
cfg.save(os.path.join(cfg.output_dir, 'config.json'))
print('Dispositivo:', cfg.device)

Dispositivo: cpu


## 2. Datos, distribución de clases y partición  [FIX-01, FIX-02]

In [4]:
df = pl.load_labels(cfg)
print(f'{len(df)} imágenes originales')
dist = pl.class_distribution(df)
dist.to_csv(os.path.join(cfg.output_dir, 'distribucion_clases.csv'), index=False)
dist

2000 imágenes originales


,valence,emotion,n,%
0,Positive,Curiosidad,168,8.400
1,Positive,Diversión,283,14.200
2,Positive,Felicidad,680,34.000
3,Neutral,Indiferencia,792,39.600
4,Negative,Desagrado,43,2.200
5,Negative,Miedo,23,1.200
6,Negative,Tristeza,11,0.600


In [5]:
splits0 = pl.make_splits(df, cfg, seed=cfg.seeds[0])
pl.save_manifest(splits0, os.path.join(cfg.output_dir, 'manifiestos', f'split_seed{cfg.seeds[0]}.csv'))
pl.split_distribution(splits0)

,train,val,test
Positive,792,169,170
Neutral,554,119,119
Negative,54,12,11


## 3. Baseline de clase mayoritaria

In [6]:
majority = pl.majority_baseline(df, cfg)
print(f"Clase mayoritaria: {majority['majority_class']}  |  accuracy={majority['accuracy']:.3f}  "
      f"balanced_acc={majority['balanced_accuracy']:.3f}  macro-F1={majority['macro_f1']:.3f}")

Clase mayoritaria: Positive  |  accuracy=0.567  balanced_acc=0.333  macro-F1=0.241


## 4. Parámetros entrenables y verificación del congelamiento  [FIX-03, FIX-04]
Tabla para la columna *Trainable params* de la Tabla II (pedido de R3). Se verifica además que un paso de entrenamiento no modifica ningún tensor del backbone (pesos ni estadísticas de BatchNorm).

In [13]:
rows = []
for exp in EXPERIMENTS + ['ft50']:
    m = pl.build_model(exp, cfg)
    trainable, total = pl.count_params(m)
    rows.append({'exp': exp, 'arch': pl.ARCH_OF_EXP[exp], 'trainable_params': trainable,
                 'total_params': total, 'backbone_frozen': m.freeze})
    del m
params_table = pd.DataFrame(rows)
params_table.to_csv(os.path.join(cfg.output_dir, 'parametros_modelos.csv'), index=False)
params_table

Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /Users/miguelcarrascoudp/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>

In [ ]:
# Prueba: un paso de optimización no debe cambiar el backbone
m = pl.build_model(8, cfg)
fp = pl.backbone_fingerprint(m)
loader = pl.make_loader(splits0['train'], pl.ARCH_OF_EXP[8], cfg, batch_size=8, train=True)
x, y = next(iter(loader))
opt = pl.make_optimizer('Adam', [p for p in m.parameters() if p.requires_grad], 1e-3)
m.train()
loss = pl.F.cross_entropy(m(x.to(cfg.device)), y.to(cfg.device))
loss.backward()
opt.step()
print('Tensores del backbone modificados:', len(pl.changed_backbone_tensors(m, fp)))  # debe ser 0
del m

## 5. Búsqueda de hiperparámetros con Optuna  [FIX-04, FIX-07, FIX-13]
Cada trial construye un modelo nuevo y usa el batch size sugerido. Métrica: `cfg.selection_metric` en validación. Los trials se guardan en `resultados/optuna/` (para rehacer la Fig. 5 como nube de puntos, sin interpolar).

In [11]:
best_params = {}
for exp in EXPERIMENTS:
    print(f'--- Optuna, experimento {exp} ---')
    study = pl.run_optuna(exp, df, cfg)
    best_params[str(exp)] = study.best_params
    print(exp, study.best_params, f'val {cfg.selection_metric}={study.best_value:.4f}')

with open(os.path.join(cfg.output_dir, 'mejores_hiperparametros.json'), 'w') as f:
    json.dump(best_params, f, indent=2)

--- Optuna, experimento 1 ---


ModuleNotFoundError: No module named 'optuna'

## 6. Entrenamiento final y evaluación en test  [FIX-10, FIX-11, FIX-12]
Para cada experimento y cada semilla: modelo nuevo → entrenamiento con early stopping sobre validación → evaluación **una vez** en el test fijo (originales sin aumentar). Se comprueba que el backbone no cambió.

In [ ]:
results = []
for exp in EXPERIMENTS:
    print(f'--- Entrenamiento final, experimento {exp} ---')
    results += pl.run_final(exp, best_params[str(exp)], df, cfg)

pd.DataFrame(results).drop(columns='confusion_matrix').to_csv(
    os.path.join(cfg.output_dir, 'resultados_por_semilla.csv'), index=False)
with open(os.path.join(cfg.output_dir, 'resultados_por_semilla.json'), 'w') as f:
    json.dump(results, f, indent=2, default=float)

## 7. Baselines  (R1: comparación con otros métodos)
- Features hand-crafted (color HSV, colorido, brillo, bordes, entropía) + SVM.
- ResNet-50 con *fine-tuning* completo (CNN estándar). Hiperparámetros fijos habituales para fine-tuning; cambie `RUN_FINETUNE` a `False` si no hay tiempo de GPU.

In [ ]:
results += pl.run_svm_baseline(df, cfg)

RUN_FINETUNE = True
if RUN_FINETUNE:
    results += pl.run_final('ft50', {'lr': 1e-4, 'batch_size': 32, 'optimizer': 'Adam'}, df, cfg)

## 8. Tabla III nueva: test, media ± DE e IC95% sobre semillas

In [ ]:
summary = pl.summarize(results)
maj_row = {'exp': 'majority', 'arch': '-', 'n_seeds': 1,
           **{k: majority[k] for k in ['accuracy', 'balanced_accuracy', 'macro_f1',
                                       'f1_Positive', 'f1_Neutral', 'f1_Negative']}}
summary = pd.concat([pd.DataFrame([maj_row]), summary], ignore_index=True)
summary.to_csv(os.path.join(cfg.output_dir, 'tabla_III_test.csv'), index=False)
summary[['exp', 'arch', 'n_seeds', 'accuracy', 'accuracy_sd', 'accuracy_ci95',
         'balanced_accuracy', 'macro_f1', 'macro_f1_ci95', 'f1_Positive', 'f1_Neutral', 'f1_Negative']]

## 9. Matrices de confusión (sumadas sobre semillas)  [R3, R4]

In [ ]:
best_resnet = summary[summary.arch == 'resnet152'].sort_values('macro_f1').iloc[-1]['exp']
best_effnet = summary[summary.arch == 'efficientnet_v2_m'].sort_values('macro_f1').iloc[-1]['exp']
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, exp in zip(axes, [best_resnet, best_effnet]):
    pl.plot_confusion(pl.aggregate_confusion(results, exp), title=f'Exp. {exp}', ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, 'matrices_confusion.png'), dpi=200)
plt.show()

## 10. Entorno de hardware y software  [R1, R3]

In [ ]:
info = pl.env_info()
with open(os.path.join(cfg.output_dir, 'entorno.json'), 'w') as f:
    json.dump(info, f, indent=2)
info